In [1]:
import pandas as pd
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

import sys
sys.path.insert(0, '.')

from src.preprocessing import preprocess

In [4]:
# 데이터 로드
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

y = train['임신 성공 여부']

In [5]:
import src.preprocessing as p
print(dir(p))

['AGE_ORDER', 'CONST_COLS', 'COUNT_ORDER', 'DONOR_AGE_ORDER', 'EMBRYO_COL', 'EMBRYO_REASONS', 'ID_COL', 'LABEL_COLS', 'LabelEncoder', 'ORDINAL_COLS', 'RECODE_ZERO_COLS', 'TARGET', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'pd', 'preprocess']


In [6]:
X_full, X_test = preprocess(train, test)

In [11]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np

seeds = [42, 52, 62]

oof_total = np.zeros(len(X_full))

for seed in seeds:
    print(f"\n====== SEED {seed} ======")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    oof = np.zeros(len(X_full))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y)):
        print(f"\n🔥 Fold {fold+1}")

        X_tr, X_va = X_full.iloc[tr_idx], X_full.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = CatBoostClassifier(
            iterations=1200,
            learning_rate=0.025,
            depth=9,
            eval_metric='AUC',
            verbose=0,
            random_seed=seed
        )

        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            early_stopping_rounds=100,
            verbose=0
        )

        oof[va_idx] = model.predict_proba(X_va)[:, 1]

    auc = roc_auc_score(y, oof)
    print(f"\n🔥 SEED {seed} AUC: {auc:.5f}")

    oof_total += oof / len(seeds)

# 🔥 최종 평균
final_auc = roc_auc_score(y, oof_total)
print("\n🚀 FINAL ENSEMBLE AUC:", final_auc)


====== SEED 42 ======

🔥 Fold 1

🔥 Fold 2

🔥 Fold 3

🔥 Fold 4

🔥 Fold 5

🔥 SEED 42 AUC: 0.73967

====== SEED 52 ======

🔥 Fold 1

🔥 Fold 2

🔥 Fold 3

🔥 Fold 4

🔥 Fold 5

🔥 SEED 52 AUC: 0.73961

====== SEED 62 ======

🔥 Fold 1

🔥 Fold 2

🔥 Fold 3

🔥 Fold 4

🔥 Fold 5

🔥 SEED 62 AUC: 0.73969

🚀 FINAL ENSEMBLE AUC: 0.7400269794985598


In [ ]:
진짜 임상 변수만 남기니 성능 너무 저하됨

In [ ]:
# # 라이브러리 & 데이터
# import pandas as pd
# import numpy as np

# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import roc_auc_score
# from catboost import CatBoostClassifier

# # 데이터 로드
# train = pd.read_csv("data/train.csv")

# # target
# y = train["임신 성공 여부"]

# # feature
# X = train.drop(["임신 성공 여부", "ID"], axis=1)

In [ ]:
# # 임상 + 핵심feature선택
# clinical_cols = [
#     # 🔹 기본
#     '시술 당시 나이',

#     # 🔹 이력
#     '총 시술 횟수',
#     '총 임신 횟수',
#     '총 출산 횟수',
#     'IVF 시술 횟수',
#     'IVF 임신 횟수',
#     'DI 시술 횟수',

#     # 🔹 시술 정보
#     '배란 유도 유형',
#     '시술 유형',

#     # 🔹 출처
#     '난자 출처',
#     '정자 출처',

#     # 🔥 핵심 추가 (있으면 반드시 포함)
#     '총 생성 배아 수',
#     '이식 배아 수',
# ]

# # 존재하는 컬럼만 필터
# clinical_cols = [c for c in clinical_cols if c in X.columns]

# X_clin = X[clinical_cols].copy()

# print("사용 feature 개수:", len(clinical_cols))
# print(clinical_cols)

사용 feature 개수: 12
['시술 당시 나이', '총 시술 횟수', '총 임신 횟수', '총 출산 횟수', 'IVF 시술 횟수', 'IVF 임신 횟수', 'DI 시술 횟수', '배란 유도 유형', '시술 유형', '난자 출처', '정자 출처', '총 생성 배아 수']


In [ ]:
# # 🔥 추가 feature 붙이기 (완성 버전)

# extra_cols = [
#     '총 생성 배아 수',
#     '이식 배아 수',
#     '수정란 수',
#     '배아 생성 주요 이유',
#     '신선 배아 사용 여부',
#     '동결 배아 사용 여부'
# ]

# # 존재 + 중복 제거
# extra_cols = [c for c in extra_cols if c in X.columns and c not in X_clin.columns]

# # join
# if len(extra_cols) > 0:
#     X_clin = X_clin.join(X[extra_cols])

# print("추가된 컬럼:", extra_cols)
# print("현재 feature 개수:", X_clin.shape[1])

추가된 컬럼: ['배아 생성 주요 이유', '신선 배아 사용 여부', '동결 배아 사용 여부']
현재 feature 개수: 15


In [ ]:
# # 🔥 categorical (X_full 기준으로 다시 정의)

# cat_cols = [
#     '시술 유형',
#     '배란 유도 유형',
#     '난자 출처',
#     '정자 출처',
#     '배아 생성 주요 이유'
# ]

# cat_cols = [c for c in cat_cols if c in X_full.columns]

# # 문자열 변환
# for col in cat_cols:
#     X_full[col] = X_full[col].astype(str)

# # 인덱스 (X_full 기준!!)
# cat_features = [X_full.columns.get_loc(c) for c in cat_cols]

# print("cat_features:", cat_features)

cat_features: [3, 6, 52, 53, 27]


In [ ]:
# # 🔥 숫자형 변환
# for col in X_clin.columns:
#     if col not in cat_cols:
#         X_clin[col] = pd.to_numeric(X_clin[col], errors='coerce')


In [ ]:
# # 🔥 파생 변수 먼저 만들기
# if '총 임신 횟수' in X_clin.columns and '총 시술 횟수' in X_clin.columns:
#     X_clin['임신 성공률 히스토리'] = X_clin['총 임신 횟수'] / (X_clin['총 시술 횟수'] + 1)

# if 'IVF 임신 횟수' in X_clin.columns and 'IVF 시술 횟수' in X_clin.columns:
#     X_clin['IVF 성공률'] = X_clin['IVF 임신 횟수'] / (X_clin['IVF 시술 횟수'] + 1)
# # 🔥 배아 효율 (여기에 추가)--> 효과없어서 삭제
# # 🔥 임신 경험 여부 (강력 feature) --> 효과 떨어져서 삭제
# # 🔥 새 후보
# if '총 시술 횟수' in X_clin.columns and '총 임신 횟수' in X_clin.columns:
#     X_clin['시술 대비 실패 횟수'] = X_clin['총 시술 횟수'] - X_clin['총 임신 횟수']

# # 🔥 마지막에 fillna
# X_clin = X_clin.fillna(-1)

In [ ]:
# #  FULL FEATURE 시작

# X_full = X.copy()

In [ ]:
# # 🔥 숫자형 변환 (cat 제외)
# for col in X_full.columns:
#     if col not in cat_cols:
#         X_full[col] = pd.to_numeric(X_full[col], errors='coerce')

# # 🔥 categorical 안정화 (NaN 포함)
# for col in cat_cols:
#     X_full[col] = X_full[col].astype(str).fillna("Unknown")

In [66]:
# 만든 feature 파생변수

if '총 임신 횟수' in X_full.columns and '총 시술 횟수' in X_full.columns:
    X_full['임신 성공률 히스토리'] = X_full['총 임신 횟수'] / (X_full['총 시술 횟수'] + 1)

if 'IVF 임신 횟수' in X_full.columns and 'IVF 시술 횟수' in X_full.columns:
    X_full['IVF 성공률'] = X_full['IVF 임신 횟수'] / (X_full['IVF 시술 횟수'] + 1)

if '총 시술 횟수' in X_full.columns and '총 임신 횟수' in X_full.columns:
    X_full['시술 대비 실패 횟수'] = X_full['총 시술 횟수'] - X_full['총 임신 횟수']

X_full = X_full.fillna(-1)

In [67]:
# Stratified K-Fold + CatBoost (X_full 버전)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# 🔥 seed ensemble

seeds = [42, 52, 62]

oof_total = np.zeros(len(X_full))

for seed in seeds:
    print(f"\n====== SEED {seed} ======")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    oof = np.zeros(len(X_full))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y)):
        print(f"\n🔥 Fold {fold+1}")

        X_tr, X_va = X_full.iloc[tr_idx], X_full.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = CatBoostClassifier(
            iterations=800,
            learning_rate=0.03,
            depth=8,
            eval_metric='AUC',
            verbose=0,
            random_seed=seed
        )

        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_features,
            early_stopping_rounds=100,
            verbose=0
        )

        preds = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = preds

        fold_auc = roc_auc_score(y_va, preds)
        print(f"Fold {fold+1} AUC: {fold_auc:.5f}")

    auc = roc_auc_score(y, oof)
    print(f"\n🔥 SEED {seed} AUC: {auc:.5f}")

    oof_total += oof / len(seeds)

# 🔥 최종 평균
final_auc = roc_auc_score(y, oof_total)
print("\n🚀 FINAL ENSEMBLE AUC:", final_auc)


====== SEED 42 ======

🔥 Fold 1
Fold 1 AUC: 0.71862

🔥 Fold 2
Fold 2 AUC: 0.72182

🔥 Fold 3
Fold 3 AUC: 0.72252

🔥 Fold 4
Fold 4 AUC: 0.72089

🔥 Fold 5
Fold 5 AUC: 0.72026

🔥 SEED 42 AUC: 0.72078

====== SEED 52 ======

🔥 Fold 1
Fold 1 AUC: 0.71978

🔥 Fold 2
Fold 2 AUC: 0.72209

🔥 Fold 3
Fold 3 AUC: 0.72204

🔥 Fold 4
Fold 4 AUC: 0.72029

🔥 Fold 5
Fold 5 AUC: 0.72138

🔥 SEED 52 AUC: 0.72107

====== SEED 62 ======

🔥 Fold 1
Fold 1 AUC: 0.72046

🔥 Fold 2
Fold 2 AUC: 0.71953

🔥 Fold 3
Fold 3 AUC: 0.72185

🔥 Fold 4
Fold 4 AUC: 0.71997

🔥 Fold 5
Fold 5 AUC: 0.72345

🔥 SEED 62 AUC: 0.72100

🚀 FINAL ENSEMBLE AUC: 0.7211963142153075
